In [1]:
import heapq
import json

class TrafficManagementSystem:
    def __init__(self):
        self.intersections = {}
        self.roads = {}

    def add_intersection(self, id, name):
        self.intersections[id] = {"name": name, "neighbors": []}

    def add_road(self, id, start, end, distance, max_speed, current_speed):
        self.roads[id] = {
            "start": start,
            "end": end,
            "distance": distance,
            "max_speed": max_speed,
            "current_speed": current_speed
        }
        if start in self.intersections:
            self.intersections[start]["neighbors"].append(id)

    def save_network(self, filename):
        data = {"intersections": self.intersections, "roads": self.roads}
        with open(filename, 'w') as f:
            json.dump(data, f)

    def load_network(self, filename):
        with open(filename, 'r') as f:
            data = json.load(f)
            self.intersections = data["intersections"]
            self.roads = data["roads"]

    def display_network(self):
        print("\n--- Intersections ---")
        for node_id, node_info in self.intersections.items():
            print(f"ID: {node_id} | Name: {node_info['name']}")
        print("\n--- Roads ---")
        for road_id, road_info in self.roads.items():
            print(f"ID: {road_id} | {road_info['start']} -> {road_info['end']} | Dist: {road_info['distance']}km | Current Speed: {road_info['current_speed']}km/h")

    def compute_optimal_route(self, source, destination, criterion="time"):
        if source not in self.intersections or destination not in self.intersections:
            return None, float('inf')

        pq = [(0, source, [])]
        visited = set()

        while pq:
            cost, current, path = heapq.heappop(pq)

            if current in visited:
                continue
            visited.add(current)

            if current == destination:
                return path + [current], cost

            for road_id in self.intersections.get(current, {}).get("neighbors", []):
                road = self.roads[road_id]
                neighbor = road["end"]
                
                if neighbor in visited:
                    continue
                
                if criterion == "distance":
                    edge_cost = road["distance"]
                else:
                    # Fix: Avoid pushing float('inf') into the heapq structure
                    if road["current_speed"] <= 0:
                        continue 
                    edge_cost = road["distance"] / road["current_speed"]

                heapq.heappush(pq, (cost + edge_cost, neighbor, path + [current]))

        return None, float('inf')

    def analyze_traffic(self):
        print("\n--- Traffic Congestion Analysis ---")
        for road_id, road in self.roads.items():
            ratio = road["current_speed"] / road["max_speed"] if road["max_speed"] > 0 else 0
            if ratio <= 0.3:
                status = "Heavy Congestion"
            elif ratio <= 0.7:
                status = "Moderate Traffic"
            else:
                status = "Free Flow"
            print(f"Road {road_id} ({road['start']}->{road['end']}): Speed Ratio: {ratio:.2f} | Status: {status}")

    def check_network_connectivity(self):
        """Checks if the network is strongly connected (every node can reach every other node)."""
        if not self.intersections:
            return True
        
        def get_reachable_count(start_node):
            visited = set()
            queue = [start_node]
            while queue:
                current = queue.pop(0)
                if current not in visited:
                    visited.add(current)
                    for road_id in self.intersections.get(current, {}).get("neighbors", []):
                        neighbor = self.roads[road_id]["end"]
                        if neighbor not in visited:
                            queue.append(neighbor)
            return len(visited)

        # Verify reachable paths originating from all nodes to confirm unified reachability
        total_nodes = len(self.intersections)
        for node in self.intersections:
            if get_reachable_count(node) != total_nodes:
                return False
        return True

if __name__ == "__main__":
    tms = TrafficManagementSystem()
    
    choice = input("Do you want to use custom input? (yes/no): ").strip().lower()
    
    if choice in ['yes', 'y']:
        num_intersections = int(input("Enter number of intersections: "))
        for _ in range(num_intersections):
            node_id = input("Intersection ID: ").strip()
            name = input("Intersection Name: ").strip()
            tms.add_intersection(node_id, name)
            
        num_roads = int(input("Enter number of roads: "))
        for _ in range(num_roads):
            road_id = input("Road ID: ").strip()
            start = input("Start Intersection ID: ").strip()
            end = input("End Intersection ID: ").strip()
            distance = float(input("Distance (km): "))
            max_speed = float(input("Max Speed (km/h): "))
            current_speed = float(input("Current Speed (km/h): "))
            tms.add_road(road_id, start, end, distance, max_speed, current_speed)
    else:
        tms.add_intersection("A", "North Station")
        tms.add_intersection("B", "Central Square")
        tms.add_intersection("C", "South Gate")
        tms.add_intersection("D", "East Boulevard")
        
        tms.add_road("R1", "A", "B", 5.0, 50, 45)
        tms.add_road("R2", "B", "C", 10.0, 60, 15)
        tms.add_road("R3", "A", "D", 7.0, 50, 48)
        tms.add_road("R4", "D", "C", 6.0, 40, 38)
    
    tms.display_network()
    
    print("\n--- Route Planning ---")
    src = input("Enter Source Intersection ID: ").strip()
    dest = input("Enter Destination Intersection ID: ").strip()
    
    path, total_time = tms.compute_optimal_route(src, dest, criterion="time")
    if path:
        print(f"Optimal Time Route: {path} | Total Travel Time: {total_time:.2f} hours")
    else:
        print("No time-optimal route found.")
        
    path_dist, total_dist = tms.compute_optimal_route(src, dest, criterion="distance")
    if path_dist:
        print(f"Optimal Distance Route: {path_dist} | Total Distance: {total_dist:.2f} km")
    else:
        print("No distance-optimal route found.")
    
    tms.analyze_traffic()
    
    print(f"\nIs network fully reachable from every position? {tms.check_network_connectivity()}")

Do you want to use custom input? (yes/no):  no



--- Intersections ---
ID: A | Name: North Station
ID: B | Name: Central Square
ID: C | Name: South Gate
ID: D | Name: East Boulevard

--- Roads ---
ID: R1 | A -> B | Dist: 5.0km | Current Speed: 45km/h
ID: R2 | B -> C | Dist: 10.0km | Current Speed: 15km/h
ID: R3 | A -> D | Dist: 7.0km | Current Speed: 48km/h
ID: R4 | D -> C | Dist: 6.0km | Current Speed: 38km/h

--- Route Planning ---


Enter Source Intersection ID:  A
Enter Destination Intersection ID:  D


Optimal Time Route: ['A', 'D'] | Total Travel Time: 0.15 hours
Optimal Distance Route: ['A', 'D'] | Total Distance: 7.00 km

--- Traffic Congestion Analysis ---
Road R1 (A->B): Speed Ratio: 0.90 | Status: Free Flow
Road R2 (B->C): Speed Ratio: 0.25 | Status: Heavy Congestion
Road R3 (A->D): Speed Ratio: 0.96 | Status: Free Flow
Road R4 (D->C): Speed Ratio: 0.95 | Status: Free Flow

Is network fully reachable from every position? False
